# Doxa Quickstart with Local Ollama

This notebook installs `doxa-ai`, installs Ollama, downloads a local model, writes a YAML scenario configured for `provider: ollama`, and runs it with `doxa run`.

## Runtime disclaimer

- This notebook assumes a Linux runtime such as Google Colab or another Ubuntu-based Jupyter environment.
- A GPU runtime is strongly recommended. CPU-only runtimes will work much more slowly, especially once you use larger models or more agents.
- The first model download can take several minutes.
- Small models such as `qwen2.5:0.5b` are useful for smoke tests and demos, but output quality will be limited compared with larger Ollama models.

In [ ]:
# Install the Doxa package and Ollama
!python -m pip install --upgrade pip
!python -m pip install doxa-ai
!curl -fsSL https://ollama.com/install.sh | sh
!ollama --version

## Start Ollama and download a local model

The next cell starts the Ollama server inside the notebook runtime and downloads `qwen2.5:0.5b`. You can replace that model name later with a larger model if your runtime has enough GPU memory.

In [ ]:
import subprocess
import time

ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.STDOUT,
)
time.sleep(5)

!ollama pull qwen2.5:0.5b

## Write an Ollama-based YAML scenario

This cell writes a compact local scenario to `ollama_local_hormuz.yaml`. Both agents use `provider: ollama` and `model_name: qwen2.5:0.5b`.

In [ ]:
from pathlib import Path
import textwrap

scenario_path = Path("ollama_local_hormuz.yaml")
scenario_path.write_text(
    textwrap.dedent(
        """
        global_rules:
          epochs: 1
          steps: 8
          execution_mode: sequential
          maintenance:
            corn: 1
          kill_conditions:
            - resource: corn
              threshold: 0
          relation_dynamics:
            on_trade_success:
              trust_delta: 0.03
            on_trade_rejected:
              trust_delta: -0.02
            on_broadcast:
              trust_delta: 0.01
            trust_decay_rate: 0.01
            panic_decay_rate: 0.05
          relations:
            - source: player
              target: miners
              trust: 0.68
              type: neutral
            - source: miners
              target: player
              trust: 0.58
              type: neutral
          markets:
            - resource: gold
              currency: credits
              initial_price: 6.0
              min_price: 1.0
              max_price: 40.0
              clearing: per_step
            - resource: corn
              currency: credits
              initial_price: 2.4
              min_price: 0.5
              max_price: 15.0
              clearing: per_step
        world_events:
          - name: gold_spike
            type: shock
            trigger:
              tick: 4
            effect:
              market: gold
              price_multiplier: 1.25
        actors:
          - id: player
            provider: ollama
            model_name: qwen2.5:0.5b
            persona: |
              Farmer-trader. Convert gold into corn, keep enough corn for maintenance, and monetize surplus when useful.
            trading_mode: both
            initial_portfolio:
              credits: 45
              corn: 10
              gold: 5
              panic: 0.0
            constraints:
              gold:
                min: 0
              corn:
                min: 0
              credits:
                min: 0
              panic:
                min: 0
                max: 1
            operations:
              farm:
                input:
                  gold: 1
                output:
                  corn: 4
          - id: miners
            provider: ollama
            model_name: qwen2.5:0.5b
            persona: |
              Miner-merchant. Convert corn into gold, use the market when possible, and keep enough corn to continue mining.
            trading_mode: both
            initial_portfolio:
              credits: 55
              corn: 6
              gold: 16
              panic: 0.0
            constraints:
              gold:
                min: 0
              corn:
                min: 0
              credits:
                min: 0
              panic:
                min: 0
                max: 1
            operations:
              mine:
                input:
                  corn: 2
                output:
                  gold: 5
        """
    ).strip() + "\n",
    encoding="utf-8",
)

print(scenario_path.read_text(encoding="utf-8"))
print(f"\nScenario written to: {scenario_path.resolve()}")

## Run Doxa on the YAML file

The next cell launches the simulation with the CLI using the scenario generated above.

In [ ]:
!doxa run ollama_local_hormuz.yaml